# Options research notebook

**Run All** refreshes the project on this Mac. It replays the synthetic fixture, attempts one Alpaca market-data capture, and values the latest stored market snapshot. Each run reads the two databases and displays the results below. All times are UTC.

In [8]:
COLLECT_MARKET = True  # Set to False to inspect saved data offline.
ROWS = 5               # Rows per database preview and most recent completed trades.

In [9]:
from contextlib import closing, redirect_stdout
from decimal import Decimal
from io import StringIO
import json
import os
from pathlib import Path
import sqlite3
import sys
from urllib.error import HTTPError, URLError

from dotenv import dotenv_values
from IPython.display import HTML, Markdown, display
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / "options_lab/schema.sql").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the trading project folder in VS Code and run the notebook again.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from options_lab.cli import parser, run
from options_lab.reporting import closed_trade_rows

DATA = ROOT / "data"
REPORTS = ROOT / "reports"
if not isinstance(ROWS, int) or ROWS < 1:
    raise ValueError("ROWS must be a positive integer.")


def project(*command):
    args = parser().parse_args(["--data-dir", str(DATA), "--report-dir", str(REPORTS), *command])
    with redirect_stdout(StringIO()):
        return run(args)


def load_alpaca_keys():
    # Read only the credentials used by the collector. Existing shell keys also work.
    values = dotenv_values(ROOT / ".env", interpolate=False)
    names = ("APCA_API_KEY_ID", "APCA_API_SECRET_KEY")
    for name in names:
        if values.get(name):
            os.environ[name] = values[name]
    return all(os.environ.get(name) for name in names)


def show_table(title, frame, *, digits=2, empty="No rows to display."):
    display(Markdown(f"**{title}**"))
    if frame.empty:
        print(empty)
        return
    shown = frame.copy()
    for name in shown.filter(regex="_utc$"):
        shown[name] = shown[name].map(
            lambda stamp: str(stamp)[:19].replace("T", " ") if pd.notna(stamp) else "—"
        )
    for name in shown.filter(regex="_usd$"):
        shown[name] = shown[name].map(
            lambda amount: f"{Decimal(str(amount)):,.2f}" if pd.notna(amount) else "—"
        )
    shown.columns = [c.replace("_", " ").replace("utc", "UTC").replace("usd", "USD")
                     .replace("pnl", "P&L") for c in shown.columns]
    table = shown.to_html(index=False, border=0, na_rep="—",
                          float_format=lambda n: f"{n:,.{digits}f}")
    display(HTML(f'<div style="overflow-x:auto;white-space:nowrap">{table}</div>'))

## Project status

A capture records when the collector received data. The option quote time comes from the feed. Those clocks can differ, especially outside market hours.

In [10]:
project("init-db")
replay_dir = project("demo")
if not COLLECT_MARKET:
    collection_status = "Offline mode. Showing stored captures."
elif not load_alpaca_keys():
    collection_status = "Alpaca keys are missing. Showing stored captures."
else:
    try:
        project("collect", "--count", "1")
    except (ValueError, OSError, KeyError, sqlite3.Error) as error:
        cause = error.__cause__
        if isinstance(cause, HTTPError):
            reason = f"Alpaca returned HTTP {cause.code}"
        elif isinstance(cause, (URLError, TimeoutError)):
            reason = "The Alpaca endpoint could not be reached"
        else:
            reason = f"The capture failed validation or storage ({type(error).__name__})"
        collection_status = f"{reason}. Showing previously stored captures."
    else:
        collection_status = "One new Alpaca capture was saved."
print(collection_status)

The capture failed validation or storage (AlpacaError). Showing previously stored captures.


In [11]:
quote_select = """
    SELECT q.capture_id, c.symbol, q.bid, q.ask, q.quote_at AS quote_utc,
           ROUND((julianday(s.retrieved_at) - julianday(q.quote_at)) * 86400, 1) AS age_at_capture_s,
           q.ingest_flags
    FROM quotes q JOIN snapshots s USING (capture_id) JOIN contracts c USING (terms_id)"""
quote_order = " ORDER BY s.retrieved_at DESC, s.capture_id DESC, c.symbol LIMIT ?"
queries = {
    "status": """
        SELECT COUNT(*) AS captures, (SELECT COUNT(*) FROM contracts) AS contract_versions,
               (SELECT COUNT(*) FROM quotes) AS quote_rows,
               (SELECT COUNT(*) FROM quotes WHERE ingest_flags <> '[]') AS flagged_rows,
               MAX(retrieved_at) AS last_capture_utc,
               (SELECT MAX(quote_at) FROM quotes) AS newest_option_quote_utc
        FROM snapshots""",
    "snapshots": """
        SELECT capture_id, retrieved_at AS retrieved_utc, underlying, spot, underlying_at AS underlying_utc,
               provider, option_feed, underlying_feed, status
        FROM snapshots ORDER BY retrieved_at DESC, capture_id DESC LIMIT ?""",
    "contracts": """
        SELECT symbol, underlying, kind, strike, expiry_at AS expiry_utc, exercise_style, multiplier, currency
        FROM contracts ORDER BY expiry_at, strike, symbol, terms_id LIMIT ?""",
    "quotes": quote_select + quote_order,
    "flagged": quote_select + " WHERE q.ingest_flags <> '[]'" + quote_order,
}

parts = {name: [] for name in queries}
for dataset, filename in [("Alpaca market", "options.sqlite3"), ("Synthetic demo", "demo.sqlite3")]:
    uri = (DATA / filename).as_uri() + "?mode=ro"
    with closing(sqlite3.connect(uri, uri=True)) as db:
        db.execute("BEGIN")  # A consistent view even if a separate collector adds a capture.
        for name, sql in queries.items():
            frame = pd.read_sql_query(sql, db, params=() if name == "status" else (ROWS,))
            frame.insert(0, "dataset", dataset)
            parts[name].append(frame)

views = {}
for name, frames in parts.items():
    nonempty = [frame for frame in frames if not frame.empty]
    views[name] = pd.concat(nonempty or frames, ignore_index=True)
status, snapshots = views["status"], views["snapshots"]
contracts, quotes = views["contracts"], views["quotes"]
show_table("Stored observations", status)
flagged = views["flagged"]
if not flagged.empty:
    show_table("Recent flagged quotes", flagged[["dataset", "capture_id", "symbol",
                                                 "age_at_capture_s", "ingest_flags"]])

**Stored observations**

dataset,captures,contract versions,quote rows,flagged rows,last capture UTC,newest option quote UTC
Alpaca market,0,0,0,0,—,—
Synthetic demo,5,2,10,1,2025-02-03 16:00:02,2025-02-03 16:00:01


**Recent flagged quotes**

dataset,capture id,symbol,age at capture s,ingest flags
Synthetic demo,demo-03,DEMO-P100,"1,801.00","[""stale_option"", ""unsynchronized""]"


## Recent simulated trades

Each row is one completed long option trade from this replay. Fill prices include slippage. Net P&L includes both fees. The summary also shows the state of any open position.

In [12]:
replay_report = json.loads((replay_dir / "replay.json").read_text())
summary = pd.DataFrame([{
    "closed_trades": replay_report["closed_trades"],
    "initial_cash_usd": replay_report["initial_cash_usd"],
    "cash_usd": replay_report["cash_usd"],
    "realized_pnl_usd": replay_report["realized_pnl_usd"],
    "unrealized_pnl_usd": replay_report["unrealized_pnl_usd"],
    "equity_usd": replay_report["equity_usd"],
    "position": replay_report["position_status"],
    "open_symbol": replay_report["open_symbol"] or "—",
}])
show_table("Synthetic replay · " + replay_dir.name, summary)
recent_trades = pd.DataFrame(closed_trade_rows(replay_report["events"]))
if not recent_trades.empty:
    recent_trades = recent_trades.sort_values("exit_utc", ascending=False).head(ROWS)
    recent_trades = recent_trades[["symbol", "exit_utc", "net_pnl_usd", "entry_utc", "buy_per_share",
                                 "sell_per_share", "fees_usd", "cash_after_usd", "contracts",
                                 "multiplier", "exit_reason"]]
show_table("Completed trades · newest exit first", recent_trades,
           empty="This replay has no completed trades.")

**Synthetic replay · replay-67832adc63e78871**

closed trades,initial cash USD,cash USD,realized P&L USD,unrealized P&L USD,equity USD,position,open symbol
1,"1,000.00","1,046.70",46.70,0.00,"1,046.70",flat,—


**Completed trades · newest exit first**

symbol,exit UTC,net P&L USD,entry UTC,buy per share,sell per share,fees USD,cash after USD,contracts,multiplier,exit reason
DEMO-C100,2025-02-03 15:45:03,46.70,2025-02-03 15:15:02,1.61,2.09,1.30,"1,046.70",1,100.00,hold_elapsed


## SQLite table heads

Each table shows up to `ROWS` records per database. Snapshots and quotes start with the newest capture. Contract terms are ordered by expiry and strike. Quote age is measured at retrieval.

In [13]:
show_table("Snapshots", snapshots)
show_table("Contracts", contracts)
show_table("Quotes", quotes)

**Snapshots**

dataset,capture id,retrieved UTC,underlying,spot,underlying UTC,provider,option feed,underlying feed,status
Synthetic demo,demo-05,2025-02-03 16:00:02,DEMO,101.20,2025-02-03 16:00:00,fixture,synthetic,synthetic,complete
Synthetic demo,demo-04,2025-02-03 15:45:03,DEMO,101.00,2025-02-03 15:45:02,fixture,synthetic,synthetic,complete
Synthetic demo,demo-03,2025-02-03 15:30:02,DEMO,100.80,2025-02-03 15:30:00,fixture,synthetic,synthetic,complete
Synthetic demo,demo-02,2025-02-03 15:15:02,DEMO,100.20,2025-02-03 15:15:00,fixture,synthetic,synthetic,complete
Synthetic demo,demo-01,2025-02-03 15:00:02,DEMO,100.00,2025-02-03 15:00:00,fixture,synthetic,synthetic,complete


**Contracts**

dataset,symbol,underlying,kind,strike,expiry UTC,exercise style,multiplier,currency
Synthetic demo,DEMO-C100,DEMO,call,100.00,2025-02-21 21:00:00,european,100.00,USD
Synthetic demo,DEMO-P100,DEMO,put,100.00,2025-02-21 21:00:00,european,100.00,USD


**Quotes**

dataset,capture id,symbol,bid,ask,quote UTC,age at capture s,ingest flags
Synthetic demo,demo-05,DEMO-C100,2.15,2.25,2025-02-03 16:00:01,1.00,[]
Synthetic demo,demo-05,DEMO-P100,1.70,1.80,2025-02-03 16:00:01,1.00,[]
Synthetic demo,demo-04,DEMO-C100,2.10,2.20,2025-02-03 15:45:02,1.00,[]
Synthetic demo,demo-04,DEMO-P100,1.75,1.85,2025-02-03 15:45:01,2.00,[]
Synthetic demo,demo-03,DEMO-C100,1.90,2.00,2025-02-03 15:30:01,1.00,[]


## Latest valuations

Model premiums and the difference from the ask are in USD per share. A blank comparison means the observation failed the run’s quality checks.

In [14]:
runs = [replay_dir]
market_snapshots = snapshots.loc[snapshots["dataset"] == "Alpaca market"]
if market_snapshots.empty:
    print("No market snapshot is stored yet. The valuation rows below come from the synthetic replay.")
else:
    capture_id = market_snapshots.iloc[0]["capture_id"]
    runs.insert(0, project("value", "--capture", capture_id))

valuation_parts = []
for directory in runs:
    values = pd.read_csv(directory / "valuations.csv", dtype={"capture_id": str})
    last_id = values.sort_values(["retrieved_at_utc", "capture_id"]).iloc[-1]["capture_id"]
    valuation_parts.append(values.loc[values["capture_id"] == last_id].head(ROWS))
valuations = pd.concat(valuation_parts, ignore_index=True)
show_table("Latest capture in each report", valuations[[
    "source", "option_feed", "capture_id", "symbol", "bid", "ask",
    "model_per_share", "model_minus_ask", "quality_flags",
]], digits=4)

No market snapshot is stored yet. The valuation rows below come from the synthetic replay.


**Latest capture in each report**

source,option feed,capture id,symbol,bid,ask,model per share,model minus ask,quality flags
synthetic,synthetic,demo-05,DEMO-C100,2.1500,2.2500,2.9920,0.7420,ok
synthetic,synthetic,demo-05,DEMO-P100,1.7000,1.8000,1.6179,-0.1821,ok
